# KTM Probabilistic Gaussian Regression with Animation

This notebook builds `(theta, delta)` points from `processing_data`, fits
\(\delta \mid 	heta \sim \mathcal{N}(\mu(	heta), \sigma(	heta)^2)\), and exports
static plots + an animation over sequence order subsets.

## 1) Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from dataclasses import dataclass
from scipy.optimize import minimize



In [ ]:
# --- Self-contained KTM processing_data function ---
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd


def processing_data(
    df: pd.DataFrame,
    *,
    user_col: str = "user",
    item_col: str = "item",
    correct_col: str = "correct",
    skill_col: str = "skill",
    order_cols: list[str] | None = None,
    reduce: bool = False,
    top_skills: int = 100,
    item_quantile: float = 0.75,
    min_user_obs: int = 10,
    rank_start: int = 0,
    c: float = 0.1,
    random_state: int = 42,
    fit_intercept: bool = False,
    center_latents: bool = True,
    target_std: float | None = 1.0,
    clip_quantiles: tuple[float, float] | None = (0.01, 0.99),
) -> pd.DataFrame:
    data = df.copy()

    needed = {user_col, item_col, correct_col}
    missing = sorted(needed - set(data.columns))
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if reduce:
        if skill_col in data.columns:
            top = data[skill_col].value_counts().head(top_skills).index
            data = data[data[skill_col].isin(top)]

        item_counts = data[item_col].value_counts()
        item_thr = float(item_counts.quantile(item_quantile))
        keep_items = item_counts[item_counts >= item_thr].index
        data = data[data[item_col].isin(keep_items)]

        user_counts = data[user_col].value_counts()
        keep_users = user_counts[user_counts >= min_user_obs].index
        data = data[data[user_col].isin(keep_users)]

    data = data.dropna(subset=[user_col, item_col, correct_col]).copy()
    data[correct_col] = (pd.to_numeric(data[correct_col], errors='coerce') >= 0.5).astype('int64')

    pipe = Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ('lr', LogisticRegression(
            solver='liblinear',
            C=c,
            random_state=random_state,
            max_iter=1000,
            fit_intercept=fit_intercept,
        )),
    ])
    y = data[correct_col]
    pipe.fit(data[[user_col, item_col]], y)

    ohe = pipe.named_steps['onehot']
    lr = pipe.named_steps['lr']
    coef = lr.coef_.ravel()

    user_ids = pd.Index(ohe.categories_[0], name=user_col)
    item_ids = pd.Index(ohe.categories_[1], name=item_col)

    n_users = len(user_ids)
    n_items = len(item_ids)
    theta = pd.Series(coef[:n_users], index=user_ids, name='theta').astype(float)
    diff = pd.Series(-coef[n_users:n_users+n_items], index=item_ids, name='difficulty').astype(float)

    if center_latents:
        theta = theta - float(theta.mean())
        diff = diff - float(diff.mean())

    if target_std is not None and target_std > 0:
        th_std = float(theta.std(ddof=0))
        di_std = float(diff.std(ddof=0))
        if th_std > 0:
            theta = theta * (float(target_std) / th_std)
        if di_std > 0:
            diff = diff * (float(target_std) / di_std)

    if clip_quantiles is not None:
        q_low, q_high = clip_quantiles
        th_lo, th_hi = theta.quantile([q_low, q_high])
        di_lo, di_hi = diff.quantile([q_low, q_high])
        theta = theta.clip(lower=float(th_lo), upper=float(th_hi))
        diff = diff.clip(lower=float(di_lo), upper=float(di_hi))

    # Rank by latent values
    theta_sorted = theta.sort_values()
    diff_sorted = diff.sort_values()

    user_rank = {u: i + rank_start for i, u in enumerate(theta_sorted.index)}
    item_rank = {it: i + rank_start for i, it in enumerate(diff_sorted.index)}

    prof_lookup = {user_rank[u]: float(v) for u, v in theta_sorted.items()}
    diff_lookup = {item_rank[it]: float(v) for it, v in diff_sorted.items()}

    data[user_col] = data[user_col].map(user_rank)
    data[item_col] = data[item_col].map(item_rank)
    data['proficiency'] = data[user_col].map(prof_lookup)
    data['difficulties'] = data[item_col].map(diff_lookup)

    # Sequence features
    sort_cols = [user_col] + (order_cols or [])
    if order_cols:
        data = data.sort_values(sort_cols, kind='mergesort')
    data['sequence_position'] = data.groupby(user_col).cumcount()
    data['sequence_length'] = data.groupby(user_col)[user_col].transform('count')
    data['order_sequence'] = data['sequence_position']

    return data


## 2) Config

In [ ]:
DATA_PATH = "../pix_mapping/pix_processed.csv"
OUT_DIR = "../pix_mapping"
os.makedirs(OUT_DIR, exist_ok=True)

# Sampling config (set SAMPLE_USERS=0 to use all users)
SAMPLE_USERS = 10000
SEED = 42

# Regression config
MU_DEGREE = 1      # 1=linear, 2=quadratic
SIGMA_DEGREE = 1   # on log-sigma
MIN_OBS_PER_SUBSET = 500
SIGMA_FLOOR = 1e-4

# Animation config
FPS = 3
MAX_SCATTER_POINTS = 5000

OUT_CSV = os.path.join(OUT_DIR, "ktm_gaussian_regression_order_notebook.csv")
OUT_PLOT = os.path.join(OUT_DIR, "ktm_gaussian_regression_order_notebook.png")
OUT_GIF = os.path.join(OUT_DIR, "ktm_gaussian_regression_order_notebook.gif")

## 3) Helpers

In [ ]:
@dataclass
class FitResult:
    beta_mu: np.ndarray
    beta_sigma: np.ndarray
    nll: float
    rmse: float
    mae: float
    success: bool
    message: str


def poly_design(x: np.ndarray, degree: int) -> np.ndarray:
    cols = [np.ones_like(x)]
    for d in range(1, degree + 1):
        cols.append(x ** d)
    return np.column_stack(cols)


def fit_gaussian(theta, delta, mu_degree=1, sigma_degree=1, sigma_floor=1e-4):
    x_mu = poly_design(theta, mu_degree)
    x_sig = poly_design(theta, sigma_degree)

    beta_mu0 = np.linalg.lstsq(x_mu, delta, rcond=None)[0]
    resid0 = delta - x_mu @ beta_mu0
    beta_sig0 = np.zeros(x_sig.shape[1], dtype=float)
    beta_sig0[0] = float(np.log(np.std(resid0) + 1e-3))
    p0 = np.concatenate([beta_mu0, beta_sig0])

    k_mu = x_mu.shape[1]
    cst = 0.5 * np.log(2.0 * np.pi)

    def nll(params):
        b_mu = params[:k_mu]
        b_sig = params[k_mu:]
        mu = x_mu @ b_mu
        sigma = np.exp(x_sig @ b_sig)
        sigma = np.maximum(sigma, sigma_floor)
        z = (delta - mu) / sigma
        return float(np.mean(cst + np.log(sigma) + 0.5 * (z ** 2)))

    out = minimize(nll, p0, method="L-BFGS-B")
    p = out.x if out.success else p0

    beta_mu = p[:k_mu]
    beta_sigma = p[k_mu:]
    mu_hat = x_mu @ beta_mu
    rmse = float(np.sqrt(np.mean((delta - mu_hat) ** 2)))
    mae = float(np.mean(np.abs(delta - mu_hat)))

    return FitResult(
        beta_mu=beta_mu,
        beta_sigma=beta_sigma,
        nll=float(nll(p)),
        rmse=rmse,
        mae=mae,
        success=bool(out.success),
        message=str(out.message),
    )

## 4) Load + process data (processing\_data)

In [ ]:
usecols = ["user_id", "challenge_id", "outcome", "answer_number", "skill_id"]
df = pd.read_csv(DATA_PATH, usecols=usecols)
df = df.rename(columns={
    "user_id": "user",
    "challenge_id": "item",
    "outcome": "correct",
    "skill_id": "skill",
})

if SAMPLE_USERS > 0 and SAMPLE_USERS < df["user"].nunique():
    rng = np.random.default_rng(SEED)
    users = df["user"].dropna().unique()
    keep = set(rng.choice(users, size=SAMPLE_USERS, replace=False).tolist())
    df = df[df["user"].isin(keep)].copy()

df_proc = processing_data(
    df,
    user_col="user",
    item_col="item",
    correct_col="correct",
    skill_col="skill",
    order_cols=["answer_number"],
    reduce=False,
    rank_start=0,
)

print("rows:", len(df_proc), "users:", df_proc["user"].nunique(), "items:", df_proc["item"].nunique())
print("order min/max:", int(df_proc["order_sequence"].min()), int(df_proc["order_sequence"].max()))

## 5) Build subsets by exact order\_sequence and fit regressions

In [ ]:
rows = []
payload = []
for order_val, g in df_proc.groupby("order_sequence", sort=True):
    if len(g) < MIN_OBS_PER_SUBSET:
        continue
    theta = g["proficiency"].to_numpy(dtype=float)
    delta = g["difficulties"].to_numpy(dtype=float)
    fit = fit_gaussian(theta, delta, MU_DEGREE, SIGMA_DEGREE, SIGMA_FLOOR)

    rec = {
        "subset": f"order_{int(order_val)}",
        "order_sequence": int(order_val),
        "n_obs": len(g),
        "mu_degree": MU_DEGREE,
        "sigma_degree": SIGMA_DEGREE,
        "nll": fit.nll,
        "rmse": fit.rmse,
        "mae": fit.mae,
        "success": fit.success,
        "message": fit.message,
    }
    for i, v in enumerate(fit.beta_mu):
        rec[f"beta_mu_{i}"] = float(v)
    for i, v in enumerate(fit.beta_sigma):
        rec[f"beta_sigma_{i}"] = float(v)
    rows.append(rec)

    payload.append({
        "subset": rec["subset"],
        "order_sequence": rec["order_sequence"],
        "n_obs": rec["n_obs"],
        "nll": rec["nll"],
        "theta": theta,
        "delta": delta,
        "beta_mu": fit.beta_mu,
        "beta_sigma": fit.beta_sigma,
    })

res = pd.DataFrame(rows).sort_values("order_sequence").reset_index(drop=True)
res.to_csv(OUT_CSV, index=False)
print("subsets fit:", len(res))
res.head()

## 6) Static panel plot

In [ ]:
n = len(payload)
ncols = 2
nrows = int(np.ceil(n / ncols)) if n > 0 else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(13, 4.3 * nrows), squeeze=False)
rng = np.random.default_rng(SEED)

for idx, rec in enumerate(payload):
    ax = axes[idx // ncols, idx % ncols]
    theta = rec["theta"]
    delta = rec["delta"]

    if len(theta) > MAX_SCATTER_POINTS:
        take = rng.choice(len(theta), size=MAX_SCATTER_POINTS, replace=False)
        xs, ys = theta[take], delta[take]
    else:
        xs, ys = theta, delta

    ax.scatter(xs, ys, s=8, alpha=0.25, color="#4477AA")
    x_grid = np.linspace(theta.min(), theta.max(), 300)
    mu = poly_design(x_grid, MU_DEGREE) @ rec["beta_mu"]
    sigma = np.exp(poly_design(x_grid, SIGMA_DEGREE) @ rec["beta_sigma"])
    sigma = np.maximum(sigma, SIGMA_FLOOR)

    ax.plot(x_grid, mu, color="#CC3311", linewidth=2.0)
    ax.plot(x_grid, mu + 2*sigma, color="#228833", linestyle="--", linewidth=1.2)
    ax.plot(x_grid, mu - 2*sigma, color="#228833", linestyle="--", linewidth=1.2)

    ax.set_title(f"{rec['subset']} | n={rec['n_obs']} | nll={rec['nll']:.3f}")
    ax.set_xlabel("theta")
    ax.set_ylabel("delta")
    ax.grid(alpha=0.2)

for j in range(n, nrows * ncols):
    axes[j // ncols, j % ncols].axis("off")

plt.tight_layout()
plt.savefig(OUT_PLOT, dpi=150)
plt.show()
print("saved", OUT_PLOT)

## 7) Animation (frame per order\_sequence)

In [ ]:
if len(payload) == 0:
    print("No subsets to animate.")
else:
    payload_sorted = sorted(payload, key=lambda r: r["order_sequence"])
    global_theta_min = min(float(np.min(r["theta"])) for r in payload_sorted)
    global_theta_max = max(float(np.max(r["theta"])) for r in payload_sorted)
    global_delta_min = min(float(np.min(r["delta"])) for r in payload_sorted)
    global_delta_max = max(float(np.max(r["delta"])) for r in payload_sorted)

    fig, ax = plt.subplots(figsize=(8.5, 6.2))
    rng = np.random.default_rng(SEED)

    def draw(i):
        ax.clear()
        rec = payload_sorted[i]
        theta = rec["theta"]
        delta = rec["delta"]

        if len(theta) > MAX_SCATTER_POINTS:
            take = rng.choice(len(theta), size=MAX_SCATTER_POINTS, replace=False)
            xs, ys = theta[take], delta[take]
        else:
            xs, ys = theta, delta

        ax.scatter(xs, ys, s=10, alpha=0.25, color="#4477AA")
        x_grid = np.linspace(global_theta_min, global_theta_max, 320)
        mu = poly_design(x_grid, MU_DEGREE) @ rec["beta_mu"]
        sigma = np.exp(poly_design(x_grid, SIGMA_DEGREE) @ rec["beta_sigma"])
        sigma = np.maximum(sigma, SIGMA_FLOOR)

        ax.plot(x_grid, mu, color="#CC3311", linewidth=2.0, label="mu(theta)")
        ax.plot(x_grid, mu + 2*sigma, color="#228833", linestyle="--", linewidth=1.2, label="mu ± 2sigma")
        ax.plot(x_grid, mu - 2*sigma, color="#228833", linestyle="--", linewidth=1.2)

        ax.set_xlim(global_theta_min, global_theta_max)
        ax.set_ylim(global_delta_min, global_delta_max)
        ax.set_xlabel("theta")
        ax.set_ylabel("delta")
        ax.set_title(f"Frame {i+1}/{len(payload_sorted)} | {rec['subset']} | n={rec['n_obs']}")
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize=8)

    ani = animation.FuncAnimation(fig, draw, frames=len(payload_sorted), interval=1000/max(FPS,1))
    writer = animation.PillowWriter(fps=max(FPS,1))
    ani.save(OUT_GIF, writer=writer)
    plt.close(fig)
    print("saved", OUT_GIF)

## 8) Notes on model form

The notebook uses:

\[
\delta \mid 	heta \sim \mathcal{N}(\mu(	heta),\sigma(	heta)^2)
\]

with

\[
\mu(	heta)=eta_0+eta_1	heta\;(+eta_2	heta^2	ext{ if degree=2})
\]

\[
\log\sigma(	heta)=\gamma_0+\gamma_1	heta\;(+\gamma_2	heta^2	ext{ if degree=2})
\]

and parameters fit by Gaussian negative log-likelihood minimization.

## 9) Off-Policy Tuples: Propensity + Reward

Build per-interaction tuples with:
- Gaussian propensity density \(p(\delta\mid	heta,	ext{order})\)
- log-propensity
- reward \(= 	exttt{correct} 	imes (\delta - \delta_{\min})\)

This is the base table for off-policy evaluation/learning.

In [ ]:
OUT_PROPENSITY_CSV = os.path.join(OUT_DIR, "ktm_offpolicy_tuples_notebook.csv")

coef_cols_mu = [c for c in res.columns if c.startswith("beta_mu_")]
coef_cols_sig = [c for c in res.columns if c.startswith("beta_sigma_")]
if len(coef_cols_mu) == 0 or len(coef_cols_sig) == 0:
    raise ValueError("No beta coefficients found in `res`. Run fitting cells first.")

coef_cols_mu = sorted(coef_cols_mu, key=lambda x: int(x.split("_")[-1]))
coef_cols_sig = sorted(coef_cols_sig, key=lambda x: int(x.split("_")[-1]))

coef_map = res.set_index("order_sequence")
available_orders = np.array(sorted(coef_map.index.unique().tolist()), dtype=int)

# Nearest-order fallback for orders not fitted (e.g., low-count orders)
def nearest_order(x: int) -> int:
    idx = np.searchsorted(available_orders, x)
    idx = min(max(idx, 0), len(available_orders) - 1)
    left = max(idx - 1, 0)
    right = idx
    if abs(x - available_orders[left]) <= abs(x - available_orders[right]):
        return int(available_orders[left])
    return int(available_orders[right])

work = df_proc.copy()
work["order_model"] = work["order_sequence"].astype(int)
missing_mask = ~work["order_model"].isin(coef_map.index)
if missing_mask.any():
    work.loc[missing_mask, "order_model"] = work.loc[missing_mask, "order_model"].map(nearest_order)

params = coef_map.loc[work["order_model"].to_numpy()]
for c in coef_cols_mu + coef_cols_sig + ["subset"]:
    work[c] = params[c].to_numpy()

In [ ]:
theta = work["proficiency"].to_numpy(dtype=float)
delta = work["difficulties"].to_numpy(dtype=float)

mu_hat = np.zeros_like(theta)
for d, c in enumerate(coef_cols_mu):
    mu_hat += work[c].to_numpy(dtype=float) * (theta ** d)

log_sigma = np.zeros_like(theta)
for d, c in enumerate(coef_cols_sig):
    log_sigma += work[c].to_numpy(dtype=float) * (theta ** d)

sigma_hat = np.maximum(np.exp(log_sigma), SIGMA_FLOOR)
z = (delta - mu_hat) / sigma_hat
log_propensity = -0.5 * np.log(2.0 * np.pi) - np.log(sigma_hat) - 0.5 * (z ** 2)
propensity = np.exp(log_propensity)

reward = work["correct"].to_numpy(dtype=float) * (delta - float(np.min(delta)))

offpolicy_tuples = pd.DataFrame(
    {
        "sequence": work["user"].to_numpy(),
        "user": work["user"].to_numpy(),
        "item": work["item"].to_numpy(),
        "correct": work["correct"].to_numpy(),
        "reward": reward,
        "answer_number": work["answer_number"].to_numpy(),
        "sequence_position": work["sequence_position"].to_numpy(),
        "order_sequence": work["order_sequence"].to_numpy(),
        "sequence_length": work["sequence_length"].to_numpy(),
        "order_model": work["order_model"].to_numpy(),
        "subset": work["subset"].to_numpy(),
        "proficiency": theta,
        "difficulties": delta,
        "mu_hat": mu_hat,
        "sigma_hat": sigma_hat,
        "log_propensity": log_propensity,
        "propensity": propensity,
    }
)

offpolicy_tuples.to_csv(OUT_PROPENSITY_CSV, index=False)

print(f"saved: {OUT_PROPENSITY_CSV}")
print("rows:", len(offpolicy_tuples))
print("propensity range:", float(offpolicy_tuples["propensity"].min()), float(offpolicy_tuples["propensity"].max()))
print("reward range:", float(offpolicy_tuples["reward"].min()), float(offpolicy_tuples["reward"].max()))
offpolicy_tuples.head()